In [63]:
# If you don't have these, uncomment the next line
# !pip install numpy pandas scipy scikit-learn matplotlib

import os, json, math, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from scipy.signal import correlate

In [77]:
DATA_ROOT = Path(r"C:\Users\Amilin Shahida\OneDrive\Desktop\HAR_SmartWatch\HAR_Projekt\data\raw")  # change to your root
FS = 100                      # resample Hz
WIN_S = 3.0                   # window length (s)
STEP_S = 1.5                  # step (s)
SINGLE_LABEL = "walk"             # e.g. "walk" to tag all windows in each session; "" = disabled

# which session to use for a demo prediction at the end (one folder that contains Left/Right csvs)
PREDICT_SESSION = "data/raw/walk/session_001"  # or Path("data/raw/walk/session_001")


In [92]:
def _read_any_csv(path):
    for sep in [",",";","\t"," "]:
        try:
            df = pd.read_csv(path, sep=sep, engine="python")
            if df.shape[1] >= 4:
                return df
        except Exception:
            pass
    raise ValueError(f"Could not parse CSV: {path}")

def _to_seconds(ts):
    try:
        t = pd.to_datetime(ts, errors="coerce")
        if t.notna().any():
            return (t - t.iloc[0]).dt.total_seconds()
        return pd.to_numeric(ts, errors="coerce")
    except Exception:
        return pd.to_numeric(ts, errors="coerce")

def _standardize_columns(df):
    cols = [c.lower() for c in df.columns.astype(str)]
    df = df.copy()
    df.columns = cols
    # best guesses for common AX export names
    def pick(*cands):
        for c in cands:
            if c in df.columns: return c
        for c in df.columns:
            lc = c.lower()
            if any(k in lc for k in cands): return c
        return None

    time_col = pick("time","timestamp","t")
    out = pd.DataFrame({"t": _to_seconds(df[time_col]) if time_col else np.arange(len(df))})
    for k, cands in {
        "ax":("ax","accx","x (g)","accel x"),
        "ay":("ay","accy","y (g)","accel y"),
        "az":("az","accz","z (g)","accel z"),
        "gx":("gx","gyrox","x (deg/s)","gyro x"),
        "gy":("gy","gyroy","y (deg/s)","gyro y"),
        "gz":("gz","gyroz","z (deg/s)","gyro z"),
    }.items():
        col = pick(*cands)
        out[k] = pd.to_numeric(df[col], errors="coerce") if col else np.nan
    return out

def _clean_numeric(df):
    df = df.copy()
    for c in ["ax","ay","az","gx","gy","gz"]:
        s = pd.Series(df[c], dtype="float64").replace([np.inf,-np.inf], np.nan)
        s = s.interpolate(limit_direction="both").fillna(method="bfill").fillna(method="ffill")
        df[c] = s
    return df

def load_ax6(path, fs=100):
    import numpy as np, pandas as pd
    df_raw = pd.read_csv(path)

    # --- parse time column (first col) robustly ---
    tcol = df_raw.columns[0]
    t = pd.to_datetime(df_raw[tcol], errors="coerce")
    # drop rows with bad time
    good = t.notna()
    df_raw = df_raw.loc[good].reset_index(drop=True)
    t = t.loc[good].reset_index(drop=True)

    # numeric columns (rename to standard if needed)
    # Expect columns named ax, ay, az, gx, gy, gz (case-insensitive OK)
    def pick(name_candidates):
        for c in df_raw.columns:
            lc = c.lower()
            if any(lc == nc for nc in name_candidates):
                return c
        return None

    cols_map = {
        "ax": pick(["ax", "accx", "acc_x", "x"]),
        "ay": pick(["ay", "accy", "acc_y", "y"]),
        "az": pick(["az", "accz", "acc_z", "z"]),
        "gx": pick(["gx", "gyrox", "gyr_x", "gx(deg/s)", "gx(rad/s)"]),
        "gy": pick(["gy", "gyroy", "gyr_y", "gy(deg/s)", "gy(rad/s)"]),
        "gz": pick(["gz", "gyroz", "gyr_z", "gz(deg/s)", "gz(rad/s)"]),
    }
    # keep what we actually found
    keep = [k for k,v in cols_map.items() if v is not None]
    df = df_raw[[cols_map[k] for k in keep]].copy()
    df.columns = keep
    df = df.apply(pd.to_numeric, errors="coerce")

    # --- build a TimedeltaIndex starting at 0s ---
    t0 = t.iloc[0]
    td_seconds = (t - t0).dt.total_seconds()
    td_index = pd.to_timedelta(td_seconds, unit="s")

    # 1) DROP duplicated time labels (keep first) and sort
    dedup = ~td_index.duplicated(keep="first")
    df = df.loc[dedup].copy()
    td_index = td_index.loc[dedup]
    df = df.set_index(td_index).sort_index()

    # 2) fill NaN/Inf with linear interpolation on time index
    df = df.replace([np.inf, -np.inf], np.nan).interpolate(method="time", limit_direction="both")

    # 3) resample to exact uniform grid at fs using resample (safer than reindex)
    step = pd.Timedelta(seconds=1/fs)
    df = df.resample(step).mean().interpolate(method="time")

    # 4) rebuild an explicit 't' column in seconds (0..N-1)/fs
    df = df.reset_index(drop=True)
    df.insert(0, "t", np.arange(len(df)) / fs)

    return df



In [93]:
def align_left_right(dfL, dfR, fs=FS, max_shift_s=2.0):
    # clean (just in case)
    for c in ("ax","ay","az","gx","gy","gz"):
        for df in (dfL, dfR):
            s = pd.Series(df[c], dtype="float64").replace([np.inf,-np.inf], np.nan)
            s = s.interpolate(limit_direction="both").fillna(method="bfill").fillna(method="ffill")
            df[c] = s

    smvL = np.sqrt(dfL["ax"]**2 + dfL["ay"]**2 + dfL["az"]**2)
    smvR = np.sqrt(dfR["ax"]**2 + dfR["ay"]**2 + dfR["az"]**2)

    corr = correlate(smvL, smvR, mode="full", method="direct")
    lags = np.arange(-len(smvR)+1, len(smvL))
    maxlag = int(max_shift_s*fs)
    m = (lags >= -maxlag) & (lags <= maxlag)
    best_lag = int(lags[m][np.argmax(corr[m])])

    if best_lag > 0:
        dfR = dfR.iloc[best_lag:].reset_index(drop=True)
        dfL = dfL.iloc[:len(dfR)].reset_index(drop=True)
    elif best_lag < 0:
        lag = -best_lag
        dfL = dfL.iloc[lag:].reset_index(drop=True)
        dfR = dfR.iloc[:len(dfL)].reset_index(drop=True)
    return dfL, dfR, best_lag


In [94]:
def make_windows(n, fs=FS, win_s=WIN_S, step_s=STEP_S):
    win = int(win_s*fs); step = int(step_s*fs)
    starts = np.arange(0, max(1, n-win+1), step, dtype=int)
    return [(s, s+win) for s in starts]

def _feat_basic(x):
    return dict(
        mean=np.mean(x), std=np.std(x),
        mad=np.mean(np.abs(x - np.mean(x))),
        min=np.min(x), max=np.max(x),
        energy=np.sum(x**2)/len(x),
        iqr=np.subtract(*np.percentile(x, [75,25])),
        zcr=np.mean(np.signbit(x[:-1]) != np.signbit(x[1:])),
    )

def _bandpower(x, fs, f_lo, f_hi):
    X = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(len(x), d=1/fs)
    m = (freqs>=f_lo) & (freqs<=f_hi)
    return float(np.sum(np.abs(X[m])**2)/len(X))

def extract_features_dual(dfL, dfR, fs=FS, win_s=WIN_S, step_s=STEP_S):
    cols = ["ax","ay","az","gx","gy","gz"]
    wins = make_windows(len(dfL), fs, win_s, step_s)
    rows = []
    for (s,e) in wins:
        row = {"t0": float(dfL["t"].iloc[s]), "t1": float(dfL["t"].iloc[e-1])}
        for side, df in (("L", dfL), ("R", dfR)):
            for c in cols:
                x = df[c].iloc[s:e].to_numpy()
                for k,v in _feat_basic(x).items():
                    row[f"{side}_{c}_{k}"] = float(v)
            for band,(lo,hi) in {"bp_low":(0.2,0.8),"bp_mid":(0.8,2.5),"bp_hi":(2.5,5.0)}.items():
                row[f"{side}_acc_{band}"] = sum(_bandpower(df[a].iloc[s:e].to_numpy(), fs, lo, hi) for a in ["ax","ay","az"])
        for c in cols:
            row[f"diff_{c}_mean"] = float(dfL[c].iloc[s:e].mean() - dfR[c].iloc[s:e].mean())
        rows.append(row)
    return pd.DataFrame(rows)


In [95]:
# Find sessions
sessions = [p for p in DATA_ROOT.rglob("session_*") if p.is_dir()]
if not sessions:
    # allow single folder with CSVs
    if (DATA_ROOT.exists() and DATA_ROOT.is_dir()):
        sessions = [DATA_ROOT]
    else:
        raise SystemExit(f"No sessions under {DATA_ROOT.resolve()}")

def find_lr_csvs(sess):
    lefts  = list(sess.glob("Left_10.csv"))
    rights = list(sess.glob("Right_10.csv"))
    if not lefts or not rights:
        return None, None
    return lefts[0], rights[0]


In [96]:
# === train_rf: end-to-end training for dual-wrist AX6 HAR ====================
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from joblib import dump
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# -------------------- CONFIG --------------------
DATA_ROOT     = Path(r"C:\Users\Amilin Shahida\OneDrive\Desktop\HAR_SmartWatch\HAR_Projekt\data\raw")
FS            = 100        # resample Hz
WIN_S         = 3.0        # window length (s)
STEP_S        = 1.5        # step (s)
SINGLE_LABEL  = ""         # e.g. "walk" to force a label for all windows in a session; "" = use labels.csv or folder name
OUT_DIR       = Path("models_rf")  # where to save the model + metadata
LEFT_NAME     = "Left_10.csv"      # adjust if your filenames differ
RIGHT_NAME    = "Right_10.csv"
# ------------------------------------------------

def find_lr_csvs(sess: Path):
    """Find left/right csv inside a session folder (tolerant to case/variants)."""
    lefts  = list(sess.glob(LEFT_NAME)) or list(sess.glob("*[Ll]eft*.csv"))
    rights = list(sess.glob(RIGHT_NAME)) or list(sess.glob("*[Rr]ight*.csv"))
    if not lefts or not rights:
        return None, None
    return lefts[0], rights[0]

def discover_sessions(data_root: Path):
    """Support two layouts:
       A) data_root/<class>/session_*/Left_10.csv,Right_10.csv
       B) data_root/session_*/Left_10.csv,Right_10.csv
       Fallback: treat data_root itself as one session if it contains both CSVs.
    """
    candidates = []

    # Classed subfolders
    for p in data_root.rglob("session_*"):
        if p.is_dir():
            candidates.append(p)

    # Direct session_* under DATA_ROOT
    for p in data_root.glob("session_*"):
        if p.is_dir():
            candidates.append(p)

    sessions = []
    for s in sorted(set(candidates)):
        L, R = find_lr_csvs(s)
        if L and R:
            sessions.append(s)

    if not sessions:
        L, R = find_lr_csvs(data_root)
        if L and R:
            sessions = [data_root]
        else:
            raise SystemExit(f"No sessions with Left/Right CSVs under {data_root.resolve()}")

    print("Found sessions:")
    for s in sessions:
        print(" -", s)
    return sessions

def label_windows_from_intervals(feats_df: pd.DataFrame, intervals_df: pd.DataFrame):
    """Map each feature window to a label using mid time ∈ [start_s, end_s]."""
    mids = 0.5 * (feats_df["t0"] + feats_df["t1"])
    labs = []
    for m in mids:
        hit = (intervals_df.start_s <= m) & (m <= intervals_df.end_s)
        labs.append(intervals_df.loc[hit, "label"].iloc[0] if hit.any() else "other")
    out = feats_df.copy()
    out["label"] = labs
    return out

def build_dataset(sessions):
    X_all, y_all = [], []
    for sess in sessions:
        left, right = find_lr_csvs(sess)
        if left is None:
            print(f"[skip] no left/right in {sess}"); 
            continue

        # load + sync
        dfL = load_ax6(left, fs=FS)
        dfR = load_ax6(right, fs=FS)
        dfL, dfR, lag = align_left_right(dfL, dfR, fs=FS)

        # features per window (left+right concatenated)
        feats = extract_features_dual(dfL, dfR, fs=FS, win_s=WIN_S, step_s=STEP_S)

        # labels: prefer labels.csv; else SINGLE_LABEL; else folder name (parent of session)
        lab_path = sess / "labels.csv"
        if lab_path.exists():
            intervals = pd.read_csv(lab_path)
            # expect columns: start_s,end_s,label
            feats = label_windows_from_intervals(feats, intervals)
        elif SINGLE_LABEL:
            feats["label"] = SINGLE_LABEL
        else:
            # use parent folder (e.g., "walk") as label; if parent is "raw", fall back to session name
            parent = sess.parent.name.lower()
            label = parent if parent not in ("raw", "data") else sess.name.lower()
            feats["label"] = label

        y_all.extend(feats["label"].values.tolist())
        X_all.append(feats.drop(columns=["label", "t0", "t1"]))

    if not X_all:
        raise SystemExit("No feature windows found. Check your data paths and file names.")
    X = pd.concat(X_all, axis=0).reset_index(drop=True)
    y = np.array(y_all)
    print("Feature matrix:", X.shape, "Classes:", sorted(set(y)))
    return X, y

def train_and_report(X, y, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    classes = sorted(np.unique(y))
    if len(classes) < 2:
        print("⚠ Only one class present:", classes[0], "→ skipping train/test split.")
        print(pd.Series(y).value_counts())

        # Train on all data so you still get a model (no evaluation possible)
        clf = RandomForestClassifier(
            n_estimators=400, class_weight="balanced", n_jobs=-1, random_state=0
        ).fit(X, y)

        dump(clf, out_dir/"rf_dual.pkl")
        X.columns.to_series().to_csv(out_dir/"rf_dual_featnames.csv", index=False)
        with open(out_dir/"rf_dual_labels.txt", "w") as f:
            for c in classes: f.write(str(c) + "\n")
        print("Saved model to", out_dir/"rf_dual.pkl")
        return clf, None, None, None
    else:
        # proper split with stratify
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.20, stratify=y, random_state=42
        )
        clf = RandomForestClassifier(
            n_estimators=400, class_weight="balanced", n_jobs=-1, random_state=0
        ).fit(Xtr, ytr)

        yp = clf.predict(Xte)
        print("\nClassification report:\n",
              classification_report(yte, yp, zero_division=0))

        # confusion matrix (pure matplotlib)
        labels_sorted = sorted(np.unique(y))
        cm = confusion_matrix(yte, yp, labels=labels_sorted)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_sorted)
        disp.plot(xticks_rotation=45, colorbar=True)
        plt.title("Confusion matrix (RF)")
        plt.tight_layout(); plt.show()

        # save artifacts
        dump(clf, out_dir/"rf_dual.pkl")
        X.columns.to_series().to_csv(out_dir/"rf_dual_featnames.csv", index=False)
        with open(out_dir/"rf_dual_labels.txt", "w") as f:
            for c in labels_sorted: f.write(str(c) + "\n")
        meta = {
            "fs": FS, "win_s": WIN_S, "step_s": STEP_S,
            "n_features": int(X.shape[1]), "classes": labels_sorted
        }
        with open(out_dir/"rf_dual_meta.json", "w") as f:
            json.dump(meta, f, indent=2)
        print("Saved model to", out_dir/"rf_dual.pkl")
        return clf, Xte, yte, yp

# -------------------- run it --------------------
sessions = discover_sessions(DATA_ROOT)
X, y = build_dataset(sessions)
clf, Xte, yte, yp = train_and_report(X, y, OUT_DIR)


Found sessions:
 - C:\Users\Amilin Shahida\OneDrive\Desktop\HAR_SmartWatch\HAR_Projekt\data\raw\session_001


KeyError: 'ax'

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np, matplotlib.pyplot as plt, seaborn as sns

labels_sorted = sorted(np.unique(y))
cm = confusion_matrix(yte, yp, labels=labels_sorted)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=labels_sorted, yticklabels=labels_sorted)
plt.title("Confusion matrix (RF)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()



NameError: name 'y' is not defined

In [87]:
# pick a session to predict (if None, use the first valid one)
if PREDICT_SESSION is None:
    PREDICT_SESSION = next((s for s in sessions if all(find_lr_csvs(s))), None)

left, right = find_lr_csvs(PREDICT_SESSION)
dfL = load_ax6(left, fs=FS); dfR = load_ax6(right, fs=FS)
dfL, dfR, _ = align_left_right(dfL, dfR, fs=FS)
feats_pred = extract_features_dual(dfL, dfR, fs=FS, win_s=WIN_S, step_s=STEP_S)
X_pred = feats_pred.drop(columns=["t0","t1"]).reindex(columns=X.columns, fill_value=0)

y_pred = rf.predict(X_pred)
timeline = feats_pred[["t0","t1"]].copy()
timeline["label"] = y_pred
timeline.head()


AttributeError: 'str' object has no attribute 'glob'

In [75]:
lbls = sorted(set(y_pred))
cmap = {lbl: plt.cm.tab20(i % 20) for i, lbl in enumerate(lbls)}

plt.figure(figsize=(10,1.4))
for _, r in timeline.iterrows():
    plt.plot([r.t0, r.t1], [0,0], lw=6, color=cmap[r.label])
plt.yticks([])
plt.xlabel("Time (s)")
plt.title(f"Predicted timeline — {PREDICT_SESSION}")
# legend
handles = [plt.Line2D([0],[0], color=cmap[l], lw=6) for l in lbls]
plt.legend(handles, lbls, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()


NameError: name 'y_pred' is not defined

In [76]:
plt.figure(figsize=(10,3))
plt.plot(dfL["t"], np.sqrt(dfL["ax"]**2+dfL["ay"]**2+dfL["az"]**2), label="SMV Left")
plt.plot(dfR["t"], np.sqrt(dfR["ax"]**2+dfR["ay"]**2+dfR["az"]**2), label="SMV Right", alpha=0.7)
plt.legend(); plt.xlabel("Time (s)"); plt.ylabel("g (approx)")
plt.title("Signal magnitude (left/right)"); plt.tight_layout(); plt.show()


NameError: name 'dfL' is not defined

<Figure size 1000x300 with 0 Axes>